# Reader note

This notebook is part of the `arXiv:2606.04091` reproduction workflow. It reproduces `Figure 1` which shows contour of $v_{0,TSA}$ and $v_{0,Emp}$ obtained from the rms-matching condition in Eq(3.5):
$$<v^2>_{SHM}(v_c,v_{esc}) = <v^2>_{Tsa}(v_{0,Tsa},v_{esc})+<v^2>_{Emp}(v_{0,Emp},v_{esc})$$
The notebook takes input `output_contour.xlsx`, which flags no-finite-solution region in case of Empirical velocity distribution and can be calulated using `rms_matching_emp_check.nb`. Old execution outputs are intentionally cleared for release.

In [ ]:
import numpy as np
import script_helpers.script_VDF as vdf
import itertools
import numpy.ma as ma
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Velocity parameters (Conservative)
# v0 = 220  
# v_esc = 544  

# Ranges for (Conservative) uncertainty bands
v0_range = [200, 280] 
v_esc_range = [450, 600] 

# Scanning ranges for v0 and v_esc 
v0_list = np.arange(v0_range[0], v0_range[1], 0.5)
v_esc_list = np.arange(v_esc_range[0], v_esc_range[1], 0.94)

v_min = 0  # lower limit
v = np.linspace(v_min, 800, 1000) # integration parameters
p_val = 1.5  # p value for empirical distribution

print(len(v0_list), len(v_esc_list)) # want to have same length for both lists to avoid issues with the meshgrid
# print(f"v0_list: {v0_list}")
# print(f"v_esc_list: {v_esc_list}")

In [ ]:
# Prepare storage for matched velocities
matched_results = []

# Loop over all combinations
for v_esc0,v00 in itertools.product(v_esc_list, v0_list):
    # Target rms value for matching is set to be equal to the rms of the MB distribution at v00 and v_esc0d
    trgt_rms = vdf.objective_MB(v00, v, v_min, v_esc0)  
    v0_MB_matched = v00  
    v0_Tsallis_matched, diff_tsallis = vdf.match_v0_Tsallis(trgt_rms, v_esc0, v, v_min, bracket=(100, 450), n_scan=500)
    v0_Empirical_matched, diff_empirical = vdf.match_v0_Empirical(trgt_rms, v_esc0, v, v_min, p=p_val, bracket=(100.0, 15000.0))

    # Append to results
    matched_results.append({
        'v0': v00,
        'v_esc': v_esc0,
        'v_rms': trgt_rms,
        'v0_MB_matched': v0_MB_matched,
        'v0_Tsallis_matched': v0_Tsallis_matched,
        'delta_rms_TSA': diff_tsallis,
        'v0_Empirical_matched': v0_Empirical_matched,
        'delta_rms_EMP': diff_empirical
    })

# # Display results
# print('Matched Velocities for Different v0 and v_esc Combinations such that v_rms  = v0')
# print(f"{'v0':>8} {'v_esc':>8} {'v_rms':>8}{'v0_MB_matched':>16} {'v0_Tsallis_matched':>20} {'delta_rms_TSA':>20} {'v0_Empirical_matched':>24} {'delta_rms_EMP':>20}")
# print("-" * 128)
# for result in matched_results:
#     print(f"{result['v0']:8.2f} {result['v_esc']:8.2f} {result['v_rms']:8.2f} {result['v0_MB_matched']:16.2f} "
#           f"{result['v0_Tsallis_matched']:20.2f} {result['delta_rms_TSA']:20.2f} "
#           f"{result['v0_Empirical_matched']:24.2f} {result['delta_rms_EMP']:20.2f}")

In [ ]:
# ---- 1. Extract sorted unique axes ----
v0_vals   = np.sort(np.unique([r["v0"] for r in matched_results]))
vesc_vals = np.sort(np.unique([r["v_esc"] for r in matched_results]))

# ---- 2. Meshgrid (physics indexing)
V0, VESC = np.meshgrid(v0_vals, vesc_vals, indexing="ij")

# ---- 3. Initialize arrays
n_v0   = len(v0_vals)
n_vesc = len(vesc_vals)

v0_tsa = np.full((n_v0, n_vesc), np.nan)
v0_emp = np.full((n_v0, n_vesc), np.nan)

# ---- 4. Index lookup
v0_index   = {v: i for i, v in enumerate(v0_vals)}
vesc_index = {v: j for j, v in enumerate(vesc_vals)}

# ---- 5. Fill arrays
for r in matched_results:
    i = v0_index[r["v0"]]
    j = vesc_index[r["v_esc"]]

    v0_tsa[i, j] = r["v0_Tsallis_matched"]

    emp_val = r["v0_Empirical_matched"]
    v0_emp[i, j] = emp_val if np.isfinite(emp_val) else np.nan

In [ ]:
# Mask invalid EMP values so contour ignores NaNs
v0_emp_masked = ma.masked_invalid(v0_emp)

# ---- FORCE COLOR RANGE ----
vmin = 100
vmax = 450
# ---- TSA (filled contour) ----
levels_tsa = np.linspace(vmin, vmax, 40)

# ---- EMP (dashed contours, extended range) ----
levels_emp = [ 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200, 220, 240, 260, 
              280, 300, 325, 350, 375, 400, 450, 500, 600, 700, 800, 900, 1000, 1500, 
              2000, 2500, 3000, 3500, 4000, 5000, 6000 ]

In [ ]:
data = pd.read_excel("output_contour.xlsx")
# print("Columns:", data.columns.tolist())
# print("\nFirst 10 rows:")
# print(data.head(10))

# Ensure it's boolean (safe conversion)
data["MB > max?"] = (
    data["MB > max?"]
    .astype(str)
    .str.strip()
    .str.upper()
    .map({"TRUE": True, "FALSE": False})
)

# Filter only True rows
true_points = data[data["MB > max?"] == True]

In [ ]:
## Making the matplotlib plots look nicer
settings = {
    # 'figure.constrained_layout.use': True,
    # 'mathtext.fontset': 'stix',
    # 'font.family': 'STIXGeneral',
    # LaTeX-like fonts
    'mathtext.fontset': 'cm',
    # 'font.family': 'serif',
    # 'font.serif': ['Computer Modern Roman'],
    # 'mathtext.fontset': 'dejavuserif',
    'font.family': 'DejaVu Serif',
    'font.size':12,
    # 'axes.labelsize': 'large',
    'lines.markersize': 5,
    'axes.linewidth':2.0,
    'xtick.major.size':8.0,
    'xtick.minor.size':4.0,
    'xtick.major.width':1.5,
    'xtick.minor.width':1.0,
    'xtick.direction':'in', 
    'xtick.minor.visible':True,
    'xtick.top':True,
    'ytick.major.size':8.0,
    'ytick.minor.size':4.0,
    'ytick.major.width':1.5,
    'ytick.minor.width':1.0,
    'ytick.direction':'in', 
    'ytick.minor.visible':True,
    'ytick.right':True,
    'contour.linewidth':3.0,
    'savefig.bbox': 'tight',
    'savefig.dpi': 200,
}

plt.rcParams.update(**settings) 

In [ ]:
plt.figure(figsize=(8,5))
cmap_tsa = "viridis"
cmap_emp = "cividis"   # or "plasma", "inferno", "cividis", etc.

# Filled TSA (explicit levels -> colorbar will reflect 100..450)
cs1 = plt.contourf(
    V0, VESC, v0_tsa,
    levels=levels_tsa,
    cmap=cmap_tsa,
    alpha=0.6
)

# EMP dashed contours (use same levels but fewer drawn)
cs2 = plt.contour(
    V0, VESC, v0_emp_masked,
    levels=levels_emp[::3],
    cmap=cmap_emp,
    linestyles="--",
    linewidths=1.5
)
plt.clabel(cs2, fmt="%.0f", fontsize=10, colors="black", rightside_up=True)

plt.scatter(
    true_points["v_0"],
    true_points["v_esc"],
    marker='o',
    color='grey',
    s=2,
    label=r'$v_{0,emp} = \infty$'
)

plt.xlabel(r"$v_\mathrm{c}$ ($\mathrm{km} \, \mathrm{s}^{-1}$)",fontsize=20)
plt.ylabel(r"$v_{\rm esc}$ ($\mathrm{km} \, \mathrm{s}^{-1}$)",fontsize=20)
plt.xlim(v0_list[0], v0_list[-1])
plt.ylim(v_esc_list[0], v_esc_list[-1])

# Colorbar
cbar1 = plt.colorbar(cs1, pad=0.02)
cbar1.set_ticks(np.arange(vmin, vmax+1, 25))
plt.tight_layout()
plt.show()